Patch manipulations basics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from yeti_iga.future.bspline import (BSpline, BSplineSurface, ControlPointManager,
    Patch, HRefiner, SubdivisionRefiner
)

Create a 2D patch

In [ ]:
# Create a set of control points
mgr = ControlPointManager(dim=2)
mgr.add_point([0.0, 1.0])
mgr.add_point([1.0, 0.0])
mgr.add_point([0.0, 2.0])
mgr.add_point([2.0, 0.0])
mgr.add_point([1.0, 1.0])
mgr.add_point([2.0, 2.0])

# Plot points with their indices in CP manager
def plot_control_points(coords):
    fig, ax = plt.subplots()

    ax.scatter(coords[:, 0], coords[:, 1], zorder=3)
    for i, (x, y) in enumerate(coords):
        ax.annotate(str(i), (x, y), textcoords="offset points", xytext=(6, 6))

    ax.set_aspect('equal')
    ax.grid(True)
    plt.show()

coords = mgr.coords_view()  # shape (n_points, 2)
plot_control_points(coords)


In [ ]:
# Create a 2D BSpline parametric space
# (degree 2 in direction 1, degree 1 in direction 2)
su = BSpline(2, np.array([0., 0., 0., 1., 1., 1.]))
sv = BSpline(1, np.array([0., 0., 1., 1.]))
surf = BSplineSurface(su, sv)

# Set mapping to control points
mapping = np.array([0, 4, 1, 2, 5, 3], dtype = np.int64)
# Number of functions in each parametric direction
local_shape = [3, 2]

# Create patch
patch = Patch(surf, mgr, mapping.tolist(), local_shape)


Evaluate patch

In [ ]:
# Evaluate 10 points at several points
n_points = 10
u = np.random.rand(n_points, 2)

spans = np.array([surf.find_span_nd(pt) for pt in u])
# Serial evaluation of point
res_serial = patch.evaluate_patch_nd(spans, u)
# Parallel evaluation of points
res_omp = patch.evaluate_patch_nd_omp(spans, u)

for pt_param, pt_span, pt_serial, pt_omp in zip(u, spans, res_serial, res_omp):
    print(f'{pt_param}, span: {pt_span}, serial: {pt_serial}, OMP: {pt_omp}')

In [ ]:
# Plot patch
def plot_2D_patch(patch, n_samples=100):


    su = patch.tensor.components[0]
    sv = patch.tensor.components[1]

    # Get elements borders
    u_breaks = np.unique(su.knot_vector)
    v_breaks = np.unique(sv.knot_vector)

    fig, ax = plt.subplots()

    # iso-u lines
    for u_val in u_breaks:
        vs = np.linspace(v_breaks[0], v_breaks[-1], n_samples)
        spans = np.array([[su.find_span(u_val), sv.find_span(v)] for v in vs], dtype=np.int32)
        params = np.column_stack([np.full(n_samples, u_val), vs])
        pts = patch.evaluate_patch_nd_omp(spans, params)        # (n_pts, dim_phys)
        ax.plot(pts[:, 0], pts[:, 1], 'b-', lw=1)

    # iso-v lines
    for v_val in v_breaks:
        us = np.linspace(u_breaks[0], u_breaks[-1], n_samples)
        spans = np.array([[su.find_span(u), sv.find_span(v_val)] for u in us], dtype=np.int32)
        params = np.column_stack([us, np.full(n_samples, v_val)])
        pts = patch.evaluate_patch_nd_omp(spans, params)
        ax.plot(pts[:, 0], pts[:, 1], 'b-', lw=1)


    ax.set_aspect('equal')
    ax.grid(True)
    plt.show()

plot_2D_patch(patch)

Refinement

In [ ]:
# Knot insertion in 1st parametric direction
T = HRefiner(direction=0, knot=0.33).refine(patch)

# Visu
plot_2D_patch(patch)
coords = mgr.coords_view()  # shape (n_points, 2)
plot_control_points(coords)

In [ ]:
# Knot insertion in 2nd parametric direction
T = HRefiner(direction=1, knot=0.66).refine(patch)

# Visu
plot_2D_patch(patch)
coords = mgr.coords_view()  # shape (n_points, 2)
plot_control_points(coords)

In [ ]:
# Subdivide span in bth directions
T = SubdivisionRefiner(direction=0, n_levels=3).refine(patch)
T = SubdivisionRefiner(direction=1, n_levels=3).refine(patch)

# Visu
plot_2D_patch(patch)
coords = mgr.coords_view()  # shape (n_points, 2)
plot_control_points(coords)

print(f'Patch have {mgr.n_points} control points')